In [1]:
import librosa
import torch
from silero_vad import load_silero_vad, get_speech_timestamps

audio_path = "../../data/test/test_audio.mp3"
sample_rate = 16000

model = load_silero_vad()
audio, _ = librosa.load(audio_path, sr=sample_rate, mono=True)
wav = torch.from_numpy(audio)

speech_timestamps = get_speech_timestamps(
    wav, model, sampling_rate=sample_rate
)
print(speech_timestamps)

/home/intern/book_audio_alignment/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[{'start': 179232, 'end': 199648}, {'start': 202784, 'end': 289248}, {'start': 295968, 'end': 331232}, {'start': 395296, 'end': 416224}, {'start': 417824, 'end': 446432}, {'start': 458272, 'end': 504288}, {'start': 521248, 'end': 570336}, {'start': 577056, 'end': 591840}, {'start': 597536, 'end': 655328}, {'start': 662560, 'end': 676832}, {'start': 695328, 'end': 725472}, {'start': 730144, 'end': 806368}, {'start': 811552, 'end': 864736}, {'start': 869408, 'end': 875488}, {'start': 879648, 'end': 886752}, {'start': 898080, 'end': 911328}, {'start': 917536, 'end': 946656}, {'start': 954400, 'end': 1000928}, {'start': 1005088, 'end': 1048544}, {'start': 1053216, 'end': 1100256}, {'start': 1109536, 'end': 1147360}, {'start': 1168928, 'end': 1230816}, {'start': 1232416, 'end': 1273312}, {'start': 1285664, 'end': 1462752}, {'start': 1482784, 'end': 1694176}, {'start': 1711648, 'end': 1815008}, {'start': 1820704, 'end': 1949664}, {'start': 1956384, 'end': 2040800}, {'start': 2043936, 'end': 

In [2]:
speech_regions = get_speech_timestamps(
    wav,
    model,
    sampling_rate=sample_rate,
    return_seconds=True,
    min_silence_duration_ms=300,
    speech_pad_ms=200
)

In [3]:
print("Speech regions:", len(speech_regions))

for region in speech_regions[:10]:
    print(region)

Speech regions: 110
{'start': 11.0, 'end': 18.2}
{'start': 18.3, 'end': 20.9}
{'start': 24.5, 'end': 28.3}
{'start': 28.5, 'end': 31.7}
{'start': 32.4, 'end': 35.8}
{'start': 35.9, 'end': 37.2}
{'start': 37.2, 'end': 41.1}
{'start': 41.2, 'end': 42.5}
{'start': 43.3, 'end': 45.5}
{'start': 45.5, 'end': 50.6}


In [4]:
from IPython.display import Audio, display

region = speech_regions[0]

start = int(region["start"] * 16000)
end = int(region["end"] * 16000)

segment = wav[start:end]

display(Audio(segment.numpy(), rate=16000))

In [5]:
import soundfile as sf
import json
from pathlib import Path

MAX_DURATION = 60
OVERLAP = 0.5

audio_duration = len(wav) / sample_rate

# Find possible splitting points between speech regions
pause_times = []

for i in range(len(speech_regions) - 1):
    current_end = speech_regions[i]["end"]
    next_start = speech_regions[i + 1]["start"]

    if next_start > current_end:
        midpoint = (current_end + next_start) / 2
        pause_times.append(midpoint)


# Step 1: Create continuous sections
sections = []
start = 0.0

while start < audio_duration:

    target = min(start + MAX_DURATION, audio_duration)

    nearby_pauses = [
        pause for pause in pause_times
        if target - 10 <= pause <= target + 10
        and pause > start
    ]

    if nearby_pauses and target < audio_duration:
        end = min(
            nearby_pauses,
            key=lambda pause: abs(pause - target)
        )
    else:
        end = target

    sections.append({
        "start": start,
        "end": end
    })

    start = end


# Step 2: Save sections with overlap
output_dir = Path("temporary_sections")
output_dir.mkdir(exist_ok=True)

saved_sections = []

for i, section in enumerate(sections):

    start_time = max(0, section["start"] - OVERLAP)
    end_time = min(audio_duration, section["end"] + OVERLAP)

    start_sample = int(start_time * sample_rate)
    end_sample = int(end_time * sample_rate)

    clip = wav[start_sample:end_sample]

    filename = f"section_{i:03d}.wav"

    sf.write(output_dir / filename, clip.numpy(), sample_rate)

    saved_sections.append({
        "filename": filename,
        "start": start_time,
        "end": end_time
    })


# Step 3: Save timestamps
with open("temporary_sections.json", "w", encoding="utf-8") as f:
    json.dump(saved_sections, f, indent=2)

print("Saved sections:", len(saved_sections))

Saved sections: 10


In [6]:
print("Total sections:", len(saved_sections))

for i in range(1, len(saved_sections)):
    previous = saved_sections[i - 1]
    current = saved_sections[i]

    overlap = previous["end"] - current["start"]

    print(f"Sections {i-1} and {i}: {overlap:.2f}s overlap")

    assert overlap >= 0, "Gap detected!"

assert saved_sections[0]["start"] == 0
assert saved_sections[-1]["end"] == audio_duration

Total sections: 10
Sections 0 and 1: 1.00s overlap
Sections 1 and 2: 1.00s overlap
Sections 2 and 3: 1.00s overlap
Sections 3 and 4: 1.00s overlap
Sections 4 and 5: 1.00s overlap
Sections 5 and 6: 1.00s overlap
Sections 6 and 7: 1.00s overlap
Sections 7 and 8: 1.00s overlap
Sections 8 and 9: 1.00s overlap


In [7]:
print(saved_sections)

[{'filename': 'section_000.wav', 'start': 0, 'end': 59.9}, {'filename': 'section_001.wav', 'start': 58.9, 'end': 122.55}, {'filename': 'section_002.wav', 'start': 121.55, 'end': 178.89999999999998}, {'filename': 'section_003.wav', 'start': 177.89999999999998, 'end': 238.65}, {'filename': 'section_004.wav', 'start': 237.65, 'end': 308.05}, {'filename': 'section_005.wav', 'start': 307.05, 'end': 365.75}, {'filename': 'section_006.wav', 'start': 364.75, 'end': 424.55}, {'filename': 'section_007.wav', 'start': 423.55, 'end': 486.1}, {'filename': 'section_008.wav', 'start': 485.1, 'end': 542.25}, {'filename': 'section_009.wav', 'start': 541.25, 'end': 591.732}]
